# Graph Store Ingestion And Exports

This notebook demonstrates property graph and RDF export surfaces, offline Neo4j/Kuzu ingestion with fake drivers, RDF-to-Neo4j neosemantics calls, Kuzu bulk import, retry policy, and dead-letter replay.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "src").exists():
            ROOT = parent
            break
sys.path.insert(0, str(ROOT / "src"))


## Build a small property graph and export it

Exporters create Cypher, Memgraph-compatible Cypher, Kuzu manifests, NetworkX node-link payloads, and RDF serializations.


In [ ]:
from pathlib import Path
import tempfile

from kg_construction.application.graph_store_benchmark import generate_graph_store_benchmark_fixture
from kg_construction.domain.graph_objects import PropertyEdge, PropertyGraph, PropertyNode, RDFGraph, RDFTriple
from kg_construction.infrastructure.graph_stores import (
    KuzuExporter,
    KuzuIngestionAdapter,
    KuzuIngestionConfig,
    MemgraphExporter,
    Neo4jExporter,
    Neo4jIngestionAdapter,
    Neo4jIngestionConfig,
    NetworkXExporter,
    RDFExporter,
    RDFLibExporter,
)
from kg_construction.infrastructure.production import BatchRetryPolicy, DeadLetterQueue, DeadLetterReplayTool, RetryPolicyConfig

sample_graph = PropertyGraph(
    nodes=[
        PropertyNode(node_id="svc:recommendation", labels=["Entity", "System"], properties={"name": "Recommendation API"}),
        PropertyNode(node_id="incident:1042", labels=["Entity", "Incident"], properties={"name": "INCIDENT-1042"}),
    ],
    edges=[PropertyEdge(edge_id="edge:affected", source_id="incident:1042", target_id="svc:recommendation", type="AFFECTED", properties={"confidence": 0.97})],
)
rdf_graph = RDFGraph(triples=[RDFTriple(subject="urn:incident:1042", predicate="urn:kg:affected", object="urn:svc:recommendation")])

exports = {
    "neo4j_statements": len(Neo4jExporter().export_property_graph(sample_graph)),
    "memgraph_statements": len(MemgraphExporter().export_property_graph(sample_graph)),
    "kuzu_nodes": len(KuzuExporter().export_property_graph(sample_graph)["nodes"]),
    "networkx_links": len(NetworkXExporter().export_property_graph(sample_graph)["links"]),
    "rdf_turtle_chars": len(RDFExporter().export_rdf_graph(rdf_graph)["turtle"]),
    "rdflib_graph_type": type(RDFLibExporter().export_rdf_graph(rdf_graph)).__name__,
}
exports


## Offline Neo4j and Kuzu ingestion

Fake drivers let the notebook prove transactional write calls, batching, constraints, and parameterized writes without needing live databases.


In [ ]:
class FakeResult:
    def consume(self):
        return None

class FakeNeo4jTx:
    def __init__(self):
        self.runs = []
    def run(self, query: str, **parameters: object):
        self.runs.append((query, parameters))
        return FakeResult()

class FakeNeo4jSession:
    def __init__(self, driver):
        self.driver = driver
        self.tx = FakeNeo4jTx()
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc, tb):
        return None
    def execute_write(self, callback, *args):
        self.driver.execute_write_calls += 1
        if self.driver.failures_before_success > 0:
            self.driver.failures_before_success -= 1
            raise RuntimeError("temporary service unavailable")
        result = callback(self.tx, *args)
        self.driver.runs.extend(self.tx.runs)
        return result

class FakeNeo4jDriver:
    def __init__(self, failures_before_success: int = 0):
        self.failures_before_success = failures_before_success
        self.execute_write_calls = 0
        self.runs = []
    def session(self, **kwargs):
        return FakeNeo4jSession(self)

class FakeKuzuConnection:
    def __init__(self):
        self.executions = []
    def execute(self, query: str, parameters: dict[str, object] | None = None):
        self.executions.append((query, parameters))
        return FakeResult()

neo_driver = FakeNeo4jDriver()
neo_report = Neo4jIngestionAdapter(
    Neo4jIngestionConfig(database="neo4j", batch_size=10),
    driver=neo_driver,
    retry_policy=BatchRetryPolicy(RetryPolicyConfig(max_attempts=1, sleep=False)),
).write_property_graph(sample_graph)

kuzu_connection = FakeKuzuConnection()
kuzu_report = KuzuIngestionAdapter(
    KuzuIngestionConfig(database_path="memory", batch_size=10),
    connection=kuzu_connection,
    retry_policy=BatchRetryPolicy(RetryPolicyConfig(max_attempts=1, sleep=False)),
).write_property_graph(sample_graph)

{
    "neo4j": neo_report.model_dump(mode="json"),
    "neo4j_query_count": len(neo_driver.runs),
    "kuzu": kuzu_report.model_dump(mode="json"),
    "kuzu_query_count": len(kuzu_connection.executions),
}


## RDF-to-Neo4j and Kuzu bulk import paths

RDF ingestion through Neo4j neosemantics is only called when explicitly configured. Kuzu bulk import writes CSV files and emits COPY statements for larger graphs.


In [ ]:
neo_rdf_driver = FakeNeo4jDriver()
neo_rdf_report = Neo4jIngestionAdapter(
    Neo4jIngestionConfig(enable_neosemantics=True),
    driver=neo_rdf_driver,
).write_rdf_graph(rdf_graph)

bulk_graph = generate_graph_store_benchmark_fixture(node_count=5, edge_factor=1)
bulk_connection = FakeKuzuConnection()
with tempfile.TemporaryDirectory() as tmpdir:
    bulk_report = KuzuIngestionAdapter(
        KuzuIngestionConfig(database_path="memory"),
        connection=bulk_connection,
    ).bulk_import_property_graph(bulk_graph, tmpdir)

{
    "neo4j_rdf": neo_rdf_report.model_dump(mode="json"),
    "neo4j_neosemantics_called": any("n10s.rdf.import.inline" in query for query, _ in neo_rdf_driver.runs),
    "kuzu_bulk_import": bulk_report.metadata["bulk_import"],
    "kuzu_copy_queries": [query for query, _ in bulk_connection.executions if query.startswith("COPY")],
}


## Retry and dead-letter replay

Failed writes can be retried, recorded to a dead-letter queue, and replayed later against a configured writer.


In [ ]:
class FakeWriter:
    name = "neo4j_ingestion_adapter"
    def __init__(self):
        self.graphs = []
    def write_property_graph(self, graph):
        self.graphs.append(graph)
        return {"written": len(graph.nodes)}
    def write_rdf_graph(self, graph):
        raise AssertionError("not used")

with tempfile.TemporaryDirectory() as tmpdir:
    queue = DeadLetterQueue(Path(tmpdir) / "dead_letters.jsonl")
    failed_report = Neo4jIngestionAdapter(
        Neo4jIngestionConfig(raise_on_failure=False),
        driver=FakeNeo4jDriver(failures_before_success=3),
        retry_policy=BatchRetryPolicy(RetryPolicyConfig(max_attempts=2, sleep=False)),
        dead_letter_queue=queue,
    ).write_property_graph(sample_graph)
    writer = FakeWriter()
    replay_report = DeadLetterReplayTool(queue).replay({"neo4j_ingestion_adapter": writer})

{
    "dead_letter_id": failed_report.dead_letter_id,
    "replayed": replay_report.replayed,
    "writer_graphs": len(writer.graphs),
}
